In [4]:
import numpy as np
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — DATA KOMPONEN MURNI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════

pure_components = {
    #   nama                  κAB              εAB/K
    "isopropylamine"   : (0.017366827,   927.4465182),
    "secbutylamine"    : (0.104540827,   629.6257827),
}

molecules = ["isopropylamine", "secbutylamine"]


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — PARAMETER REFERENSI GRUP (INPUT — SUDAH DIKETAHUI)
# ══════════════════════════════════════════════════════════════════════════════

GC_ref = {
    #  grup     κAB_ref      εAB_ref/K
    "CH3"  : (0.0,          0.0      ),   
    "CH2"  : (0.0,          0.0      ),   
    "CH"   : (0.0,          0.0      ),  
    "NH2"  : (0.149350,     504.320  ),   # dari Vijande et al. Table 1
    "OH"   : (0.001341,     2217.87  ),   # dari Vijande et al. Table 1
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — PARAMETER PERTURBATIF μ YANG SUDAH DIKETAHUI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════

mu_known = {
    "CH3-CH3"  : {"kap":  0.0,   "eps":  0.0  },   
    "CH3-CH"   : {"kap":  0.0,   "eps":  0.0  },  
    "CH2-CH3"  : {"kap":  0.0,   "eps":  0.0  },   
    "CH2-CH"   : {"kap":  0.0,   "eps":  0.0  },   
    "CH3-NH2"  : {"kap":  -0.0311,   "eps":  111.93  },   
    "CH2-NH2"  : {"kap":  0.0,   "eps":  0.0  },   
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — S-TERM (INPUT MANUAL)
# ══════════════════════════════════════════════════════════════════════════════

S_terms = {
    #                  isopropylamine   secbutylamine
    "CH3-CH3"    : [  0.5,             0.333333333  ],
    "CH3-CH"     : [  2.0,             1.5          ],
    "CH2-CH3"    : [  0.0,             1.5          ],
    "CH2-CH"     : [  0.0,             1.0          ],
    "CH3-NH2"    : [  1.0,             0.833333333  ],
    "CH2-NH2"    : [  0.0,             0.5          ],
    "CH-NH2"     : [  1.0,             1.0          ],
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — VARIABEL YANG DI-FITTING (UNKNOWN)
# ══════════════════════════════════════════════════════════════════════════════

unknown_pairs = [
    "CH-NH2",   
]

# Jumlah unknown per properti (dihitung otomatis)
N_unk = len(unknown_pairs)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — PERSAMAAN MODEL PER MOLEKUL
# ══════════════════════════════════════════════════════════════════════════════

def model(mol_idx, prop, unk):
    mu_CH_NH2 = unk[0]
    if mol_idx == 0:
        # isopropyl
        return (
            2 * gc("CH3", prop)                                    # 2×κAB_CH3 (=0, tapi tetap ditulis)
            + 1 * gc("CH",  prop)                                  # 1×κAB_CH  (=0)
            + 1 * gc("NH2", prop)                                  # 1×κAB_NH2
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)     # 2×μ(CH3-CH3)·S [known]
            + 1 * mk("CH3-CH",  prop) * S("CH3-CH",  mol_idx)     # 2×μ(CH3-CH)·S  [known] — 2 CH3 × 1 CH
            + 1 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)     # 2×μ(CH3-NH2)·S [known] — 2 CH3 × 1 NH2
            + 1 * mu_CH_NH2          * S("CH-NH2",   mol_idx)     # 2×μ(CH-NH2)·S  [unknown]
        )

    elif mol_idx == 1:
        # secbutyl
        return (
            2 * gc("CH3", prop)                                    # 2×κAB_CH3 (=0)
            + 1 * gc("CH2", prop)                                  # 1×κAB_CH2 (=0)
            + 1 * gc("CH",  prop)                                  # 1×κAB_CH  (=0)
            + 1 * gc("NH2", prop)                                  # 1×κAB_NH2
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)     # 2×μ(CH3-CH3)·S [known]
            + 1 * mk("CH3-CH",  prop) * S("CH3-CH",  mol_idx)     # 2×μ(CH3-CH)·S  [known]
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)     # 1×μ(CH2-CH3)·S [known] — 1 CH2 × 2 CH3
            + 1 * mk("CH2-CH",  prop) * S("CH2-CH",  mol_idx)     # 1×μ(CH2-CH)·S  [known]
            + 1 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)     # 2×μ(CH3-NH2)·S [known]
            + 1 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)     # 1×μ(CH2-NH2)·S [known]
            + 1 * mu_CH_NH2          * S("CH-NH2",   mol_idx)     # 2×μ(CH-NH2)·S  [unknown]
        )

    else:
        raise ValueError(f"mol_idx={mol_idx} tidak ada dalam daftar molekul.")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — BATAS (BOUNDS) UNTUK UNKNOWN
# ══════════════════════════════════════════════════════════════════════════════
bounds = [
    # kappa
    (-1.0,  1.0),
    # epsilon
    (-2000.0,  2000.0),
]

# Validasi otomatis — tidak perlu diedit
assert len(bounds) == N_unk * 2, (
    f"Jumlah bounds ({len(bounds)}) harus = N_unk×2 = {N_unk*2}. "
    f"Sesuaikan SECTION 7: {N_unk} baris untuk κAB, lalu {N_unk} baris untuk εAB."
)


# ══════════════════════════════════════════════════════════════════════════════
# Helper functions, objective function, optimizer, output
# ══════════════════════════════════════════════════════════════════════════════

# ── Helper: ambil nilai referensi grup ──────────────────────────────────────
def gc(group, prop):
    """
    Kembalikan nilai referensi κAB atau εAB untuk grup.
    prop = "kap" → κAB_grup
    prop = "eps" → εAB_grup
    """
    kap_g, eps_g = GC_ref[group]
    return {"kap": kap_g, "eps": eps_g}[prop]

# ── Helper: ambil μ known ────────────────────────────────────────────────────
def mk(pair, prop):
    """Kembalikan nilai μ yang sudah diketahui untuk pasangan dan properti."""
    if pair not in mu_known:
        raise KeyError(
            f"Pasangan '{pair}' tidak ada di mu_known (SECTION 3). "
            f"Tambahkan jika diketahui, atau masukkan ke unknown_pairs (SECTION 5)."
        )
    return mu_known[pair][prop]

# ── Helper: ambil S-term ─────────────────────────────────────────────────────
def S(pair, mol_idx):
    """Kembalikan nilai S-term untuk pasangan dan indeks molekul."""
    if pair not in S_terms:
        raise KeyError(
            f"Pasangan '{pair}' tidak ada di S_terms (SECTION 4). "
            f"Tambahkan nilainya."
        )
    return S_terms[pair][mol_idx]

# ── Hitung target dari data komponen murni ───────────────────────────────────
# targets["kap"] = array κAB tiap molekul
# targets["eps"] = array εAB tiap molekul
targets = {"kap": [], "eps": []}
for mol in molecules:
    kap, eps = pure_components[mol]
    targets["kap"].append(kap)
    targets["eps"].append(eps)
targets = {prop: np.array(v) for prop, v in targets.items()}

# ── Unpack unknown dari vektor x ─────────────────────────────────────────────
def unpack(x, prop):
    """
    Ambil slice unknown untuk properti tertentu dari vektor x optimizer.
    Urutan dalam x: [kap_unk0, kap_unk1, ..., eps_unk0, eps_unk1, ...]
    """
    i = {"kap": 0, "eps": 1}[prop]
    return x[i * N_unk : (i + 1) * N_unk]

# ── Objective function ────────────────────────────────────────────────────────
def objective(x):
    """
    F_OBJ = Σ_prop Σ_molekul [ (π_target - π_GC) / π_target ]²

    prop ∈ {kap (κAB), eps (εAB)}

    Vektor x:
      x[0 : N_unk]       → unknown untuk κAB
      x[N_unk : 2*N_unk] → unknown untuk εAB
    """
    F = 0.0
    for prop in ("kap", "eps"):
        unk = unpack(x, prop)
        for mol_idx in range(len(molecules)):
            pi_GC  = model(mol_idx, prop, unk)
            pi_tgt = targets[prop][mol_idx]
            # Hindari pembagian dengan nol jika target = 0
            if abs(pi_tgt) > 1e-15:
                F += ((pi_tgt - pi_GC) / pi_tgt) ** 2
    return F

# ── Optimasi dua tahap: global → lokal ───────────────────────────────────────
print("=" * 62)
print("  PC-SAFT PertGC — Association Parameter Fitting Template")
print("=" * 62)
print(f"\n  Molekul  : {', '.join(molecules)}")
print(f"  Unknown  : {', '.join(unknown_pairs)}")
print(f"  Total    : {N_unk} unknown × 2 properti = {N_unk*2} parameter\n")

print("  Stage 1: Global search (differential_evolution)...")
res_g = differential_evolution(
    objective, bounds,
    seed=42, maxiter=8000, tol=1e-14,
    popsize=25, mutation=(0.5, 1.5), recombination=0.9,
    polish=True, disp=False,
)

print("  Stage 2: Local refinement (L-BFGS-B)...")
res_l = minimize(
    objective, res_g.x,
    method="L-BFGS-B", bounds=bounds,
    options={"maxiter": 100000, "ftol": 1e-16, "gtol": 1e-13},
)

x_opt = res_l.x if res_l.fun < res_g.fun else res_g.x
F_opt = min(res_l.fun, res_g.fun)

# ── Cetak parameter hasil fitting ────────────────────────────────────────────
prop_label = {"kap": "κAB", "eps": "εAB (K)"}

print("\n" + "=" * 62)
print("  FITTED PARAMETERS")
print("=" * 62)
for prop in ("kap", "eps"):
    unk = unpack(x_opt, prop)
    print(f"\n  Properti: {prop_label[prop]}")
    for j, pair in enumerate(unknown_pairs):
        print(f"    μ_{prop_label[prop].split()[0]}({pair:<18}) = {unk[j]:>16.6f}")

print(f"\n  F_OBJ (final) = {F_opt:.6e}")

# ── Validasi: target vs. prediksi GC ─────────────────────────────────────────
print("\n" + "=" * 62)
print("  VALIDATION — Target vs. GC-Predicted (APD%)")
print("=" * 62)

for prop in ("kap", "eps"):
    unk = unpack(x_opt, prop)
    print(f"\n  {prop_label[prop]}")
    print(f"  {'Molekul':<20} {'Target':>14} {'GC-Pred':>14} {'APD%':>9}")
    print(f"  {'-'*60}")
    for mol_idx, mol in enumerate(molecules):
        pi_tgt = targets[prop][mol_idx]
        pi_gc  = model(mol_idx, prop, unk)
        apd    = 100.0 * abs(pi_tgt - pi_gc) / abs(pi_tgt) if abs(pi_tgt) > 1e-15 else 0.0
        print(f"  {mol:<20} {pi_tgt:>14.6f} {pi_gc:>14.6f} {apd:>8.4f}%")

# ── Tabel ringkasan ───────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  SUMMARY TABLE (copy-paste ready)")
print("=" * 62)
print(f"  {'Parameter':<28} {'κAB':>14} {'εAB (K)':>14}")
print("  " + "-" * 58)
for j, pair in enumerate(unknown_pairs):
    val_kap = unpack(x_opt, "kap")[j]
    val_eps = unpack(x_opt, "eps")[j]
    print(f"  μ({pair:<24}) {val_kap:>14.6f} {val_eps:>14.4f}")

# ── Diagnostik batas ──────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  DIAGNOSTIC — Bound Check")
print("=" * 62)

all_labels = (
    [f"μ_kap({p})" for p in unknown_pairs] +
    [f"μ_eps({p})" for p in unknown_pairs]
)

bound_hit = False
for i, (val, (lo, hi)) in enumerate(zip(x_opt, bounds)):
    tol = max(1e-4, 1e-4 * abs(hi - lo))
    if abs(val - lo) < tol or abs(val - hi) < tol:
        print(f"  ⚠  {all_labels[i]} = {val:.6f} menempel di batas [{lo}, {hi}].")
        print(f"      → Perlebar batas ini di SECTION 7 dan jalankan ulang.")
        bound_hit = True
if not bound_hit:
    print("  ✓  Tidak ada parameter yang menempel di batas. Solusi interior.")

  PC-SAFT PertGC — Association Parameter Fitting Template

  Molekul  : isopropylamine, secbutylamine
  Unknown  : CH-NH2
  Total    : 1 unknown × 2 properti = 2 parameter

  Stage 1: Global search (differential_evolution)...
  Stage 2: Local refinement (L-BFGS-B)...

  FITTED PARAMETERS

  Properti: κAB
    μ_κAB(CH-NH2            ) =        -0.098681

  Properti: εAB (K)
    μ_εAB(CH-NH2            ) =       120.102211

  F_OBJ (final) = 6.606162e-01

  VALIDATION — Target vs. GC-Predicted (APD%)

  κAB
  Molekul                      Target        GC-Pred      APD%
  ------------------------------------------------------------
  isopropylamine             0.017367       0.019569  12.6791%
  secbutylamine              0.104541       0.024752  76.3230%

  εAB (K)
  Molekul                      Target        GC-Pred      APD%
  ------------------------------------------------------------
  isopropylamine           927.446518     736.352211  20.6043%
  secbutylamine            629.625783